<h3 style="color:#6FA8DC; font-weight:bold">06 — Automatically Choosing Imputation Parameters with GridSearchCV</h3>

This notebook follows the provided CampusX `automatically-select-imputer-parameters.ipynb` reference and expands the idea into a modern leakage-safe ML workflow.

<h5 style="color:#78B89A; font-weight:bold;">1. The problem → which imputation strategy should we choose?</h5>

You may have several reasonable choices:

- numerical: mean or median
- numerical: constant value
- categorical: most frequent or constant category
- model hyperparameters such as Logistic Regression `C`

Instead of manually guessing one combination, let **cross-validation evaluate the candidate combinations**.

<h5 style="color:#78B89A; font-weight:bold;">2. Important clarification → GridSearchCV does not magically invent a method</h5>

GridSearchCV searches the candidates that **you provide**.

```text
You define candidate methods
          ↓
GridSearchCV tries combinations
          ↓
Cross-validation scores each combination
          ↓
Best CV score → best candidate
```

So it is better to say: **automatically select the best method from the candidate methods we provide.**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('titanic_toy.csv')
df.head()

df.isnull().mean() * 100

<h5 style="color:#78B89A; font-weight:bold;">3. Prepare X and y → same as normal ML workflow</h5>

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numerical_features = ['Age', 'Fare', 'Family']

<h5 style="color:#78B89A; font-weight:bold;">4. Build the preprocessing Pipeline → imputation + scaling</h5>

The key is that the imputer is **inside the Pipeline**. Therefore, during every cross-validation fold, the imputation statistic is learned only from that fold's training portion.

That prevents preprocessing leakage.

In [ ]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features)
    ]
)

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

<h5 style="color:#78B89A; font-weight:bold;">5. Define the search space → candidate imputation methods</h5>

We can ask GridSearchCV to compare:

- `mean`
- `median`
- `constant` with `-1`
- `constant` with `999`
- different Logistic Regression `C` values

The exact constants are candidates, not universal rules.

In [ ]:
param_grid = [
    {
        'preprocessor__num__imputer__strategy': ['mean', 'median'],
        'classifier__C': [0.1, 1, 10, 100]
    },
    {
        'preprocessor__num__imputer__strategy': ['constant'],
        'preprocessor__num__imputer__fill_value': [-1, 0, 999],
        'classifier__C': [0.1, 1, 10, 100]
    }
]

print('Candidate numerical strategies:', ['mean', 'median', 'constant(-1)', 'constant(0)', 'constant(999)'])

<h5 style="color:#78B89A; font-weight:bold;">6. Run GridSearchCV → cross-validation chooses the best candidate</h5>

In [ ]:
grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=10,
    scoring='accuracy',
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

print('Best parameters:')
print(grid_search.best_params_)
print('\nBest internal CV score:', round(grid_search.best_score_, 4))

<h5 style="color:#78B89A; font-weight:bold;">7. Evaluate on the untouched test set → final check</h5>

In [ ]:
y_pred = grid_search.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print('Test accuracy:', round(test_accuracy, 4))

<h5 style="color:#78B89A; font-weight:bold;">8. Inspect all combinations → understand why GridSearch chose one</h5>

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

cols = [
    'param_preprocessor__num__imputer__strategy',
    'param_preprocessor__num__imputer__fill_value',
    'param_classifier__C',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]

results = cv_results[[c for c in cols if c in cv_results.columns]] \
    .sort_values('rank_test_score')
results.head(15)

<h5 style="color:#78B89A; font-weight:bold;">9. Why Pipeline + GridSearchCV is the modern ML way → leakage-safe model selection</h5>

Without a Pipeline, you might calculate an imputation statistic on the entire training dataset before cross-validation. Then every validation fold indirectly sees information from the other folds.

With a Pipeline:

```text
Fold 1 train
   ↓
fit imputer on Fold 1 train only
   ↓
transform Fold 1 validation
   ↓
fit model
   ↓
score

Repeat for every fold
```

GridSearchCV then compares the complete workflows rather than comparing preprocessing in isolation.

<h5 style="color:#78B89A; font-weight:bold;">10. Adding missing indicators to the search → another candidate</h5>

You can also let the search compare whether a missingness indicator helps.

In [ ]:
indicator_grid = [
    {
        'preprocessor__num__imputer__strategy': ['mean', 'median'],
        'preprocessor__num__imputer__add_indicator': [False, True],
        'classifier__C': [0.1, 1, 10]
    },
    {
        'preprocessor__num__imputer__strategy': ['constant'],
        'preprocessor__num__imputer__fill_value': [-1, 0, 999],
        'preprocessor__num__imputer__add_indicator': [False, True],
        'classifier__C': [0.1, 1, 10]
    }
]

indicator_search = GridSearchCV(
    clf, indicator_grid, cv=5, scoring='accuracy', n_jobs=-1
)
indicator_search.fit(X_train, y_train)

print('Best configuration with indicator search:')
print(indicator_search.best_params_)
print('Best CV score:', round(indicator_search.best_score_, 4))

<h5 style="color:#78B89A; font-weight:bold;">11. What about Random Sample Imputation? → important limitation</h5>

The standard `SimpleImputer` does **not** provide a `random_sample` strategy. Therefore you cannot simply put `random_sample` into the same parameter list.

To include it in an automated search, you would need a custom scikit-learn-compatible transformer implementing `fit()` and `transform()` and then place that transformer inside the Pipeline.

The same principle still applies: define the candidates → cross-validation evaluates them → choose the best based on the selected scoring metric.

<h5 style="color:#78B89A; font-weight:bold;">12. What does “best” mean? → scoring metric decides it</h5>

GridSearchCV does not determine an absolute best imputation method.

It determines the candidate that gives the best **cross-validation score for the metric you chose**.

For classification, possible metrics include accuracy, precision, recall, F1 and ROC-AUC. For regression, common choices include MAE, MSE/RMSE and R².

So always choose a metric that matches the actual ML objective.

<h5 style="color:#78B89A; font-weight:bold;">13. Production flow → final architecture</h5>

```text
Raw training data
      ↓
Train / Test split
      ↓
Pipeline
  ┌──────────────────────┐
  │ Imputer candidate    │
  │ Scaling              │
  │ Model                │
  └──────────────────────┘
      ↓
GridSearchCV
      ↓
Cross-validation
      ↓
Best candidate
      ↓
Refit on full training set
      ↓
Untouched test evaluation
      ↓
Deploy best fitted Pipeline
      ↓
Production data → same preprocessing → prediction
```

<h5 style="color:#78B89A; font-weight:bold;">14. Final revision → remember these lines</h5>

- GridSearchCV **does not invent** preprocessing methods; it searches the candidates you define.
- Put preprocessing inside a `Pipeline` before GridSearchCV.
- This makes cross-validation preprocessing leakage-safe.
- `best_params_` gives the selected configuration.
- `best_score_` gives the best internal CV score.
- Use the untouched test set only after selection.
- Random sample imputation requires a custom transformer if you want it in the search.

**One-line definition:** GridSearchCV can automatically select the best imputation configuration from a predefined search space by comparing complete preprocessing + model pipelines using cross-validation.